In [10]:
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import time
from pathlib import Path
from datasets import Dataset, load_from_disk



In [11]:
torch.cuda.empty_cache()
gc.collect()
print("GPU Cache Cleared")
print(f"Available GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU Cache Cleared
Available GPU: 12.9 GB


In [12]:
verifier_id = "Qwen/Qwen2.5-7B-Instruct"
drafter_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Tokenizers
verifier_tokenizer = AutoTokenizer.from_pretrained(verifier_id)
drafter_tokenizer = AutoTokenizer.from_pretrained(drafter_id)

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, 
    #float16 instead of torch.bfloat16 used due to my gpu of RTX 2060 does not accelerate bfloat16 which is faster to load (23s vs 31s with bfloat16)
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load models
print("Loading verifier...")
verifier = AutoModelForCausalLM.from_pretrained(
    verifier_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
)

print("Loading drafter...")
drafter = AutoModelForCausalLM.from_pretrained(
    drafter_id,
    dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
)

# Verify
allocated = torch.cuda.memory_allocated() / 1e9
reserved = torch.cuda.memory_reserved() / 1e9
print(f"✓ Models loaded: {allocated:.1f}GB allocated | {reserved:.1f}GB reserved")

Loading verifier...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading drafter...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✓ Models loaded: 12.1GB allocated | 15.3GB reserved


In [13]:
print("Loading NQ dataset...")
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
NQ_DIR = ROOT / "data" / "nq_validation"
if NQ_DIR.exists():
    nq = load_from_disk(NQ_DIR)
    print(f"Loaded {len(nq)} examples from {NQ_DIR}")
else:
    stream = load_dataset(
        "google-research-datasets/natural_questions",
        split="validation",
        streaming=True,
    )
    nq = Dataset.from_list(list(stream.take(1000)))
    NQ_DIR.parent.mkdir(parents=True, exist_ok=True)
    nq.save_to_disk(NQ_DIR)
    print(f"Downloaded {len(nq)} validation examples and wrote them to {NQ_DIR}")
print(nq.column_names)

print("Encoding passages...")
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
def extract_document_text(doc: dict, max_words: int = 250) -> str:
    """Extract clean text without HTML tags, optionally truncated to a passage."""
    tokens = doc["tokens"]
    clean_tokens = [
        tok for tok, is_html in zip(tokens["token"], tokens["is_html"])
        if not is_html
    ]
    # Truncate to passage size so MiniLM (max 256 tokens) doesn't drop the rest
    if max_words:
        clean_tokens = clean_tokens[:max_words]
    return " ".join(clean_tokens)

passages = [extract_document_text(ex["document"]) for ex in nq]

# Note: normalize_embeddings=True ensures cosine similarity with IndexFlatIP
embeddings = encoder.encode(
    passages, 
    batch_size=32, 
    show_progress_bar=True, 
    normalize_embeddings=True
)
print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings.astype("float32"))

INDEX_PATH = ROOT / "data" / "nq_index_1k.faiss"
INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(INDEX_PATH))

print(f"✓ Index built: {index.ntotal} passages saved to {INDEX_PATH}")



Loading NQ dataset...
Loaded 1000 examples from c:\Users\Gerent\Documents\GitHub\ARASD\data\nq_validation
['id', 'document', 'question', 'long_answer_candidates', 'annotations']
Encoding passages...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Building FAISS index...
✓ Index built: 1000 passages saved to c:\Users\Gerent\Documents\GitHub\ARASD\data\nq_index_1k.faiss


In [17]:
def baseline_speculative_decode(query, num_draft_tokens=5):
    """Vanilla speculative decoding without retrieval."""
    
    # Draft with smaller model
    draft_inputs = drafter_tokenizer(query, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        draft_ids = drafter.generate(
            **draft_inputs,
            max_new_tokens=num_draft_tokens,
            output_scores=True,
            return_dict_in_generate=True
        )
    
    draft_tokens = draft_ids.sequences[0, len(draft_inputs.input_ids[0]):]
    
    # Verify with larger model
    verify_input = torch.cat([
        draft_inputs.input_ids,
        draft_tokens.unsqueeze(0)
    ], dim=1)
    
    with torch.no_grad():
        final_output = verifier.generate(
            verify_input,
            max_new_tokens=512,
            temperature=0.7
        )
    
    return verifier_tokenizer.decode(final_output[0], skip_special_tokens=True)

# Test
query = "What is the capital of France?"
result = baseline_speculative_decode(query)
print(result)

What is the capital of France? The capital of France is Paris. 

Paris is a city located in the northern part of the country, and it is known for its rich history, culture, and architecture. It is home to many famous landmarks such as the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral. Additionally, Paris is often referred to as the "City of Love" due to its romantic atmosphere and numerous attractions for tourists and locals alike. It serves as the political, cultural, and economic center of France. 

Would you like to know more about Paris or any other topic related to France? I'm happy to provide additional information! 😊🌍🇫🇷✍️🎨✈️❤️💕✨📊🔍📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊📊

In [18]:
def speculative_decode(query, max_new_tokens=512, num_assistant_tokens=5):
    """
    Speculative Decoding using Universal Assisted Decoding (UAD).
    - verifier: target model (Qwen2.5-7B-Instruct)
    - drafter: assistant model (Qwen2.5-1.5B-Instruct)
    """
    messages = [{"role": "user", "content": query}]
    prompt = verifier_tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    inputs = verifier_tokenizer(prompt, return_tensors="pt").to("cuda")
    prompt_len = inputs.input_ids.shape[1]
    
    with torch.no_grad():
        output = verifier.generate(
            **inputs,
            assistant_model=drafter,
            # Explicitly supply tokenizers for Universal Assisted Decoding:
            tokenizer=verifier_tokenizer,
            assistant_tokenizer=drafter_tokenizer,
            num_assistant_tokens=num_assistant_tokens,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Standard greedy evaluation for speculative decoding
            pad_token_id=verifier_tokenizer.eos_token_id,
        )
    
    # Strip off the prompt so only the generated response is returned
    response_tokens = output[0][prompt_len:]
    return verifier_tokenizer.decode(response_tokens, skip_special_tokens=True)

# Test
query = "What is the capital of France?"
result = speculative_decode(query)
print(result)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'min_new_tokens', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.RepetitionPenaltyLogitsPro

The capital of France is Paris.
